# K-mer Feature Pipeline (Binary presence)

This notebook performs k-mer feature selection and trains models using binary presence/absence
features for selection and final modeling. The prevalence-first approach keeps memory use low.
Models and vocabulary are saved to
`output/.`

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse



## Helper functions
These helpers iterate per-genome k-mer dump files, build prevalence counts, construct
sparse matrices (binary mode), and load phenotype labels.

In [2]:
# Helper utilities for k-mer pipelines (binary-presence notebook)
# Each function below includes a short comment/docstring explaining its role, inputs, and outputs.
from typing import Iterable

def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
    """Read a per-genome k-mer dump and return a dict of {kmer: count}.

    Parameters:
    - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

    Returns:
    - dict mapping kmer (str) to integer count.
    """
    kmers: dict[str, int] = {}
    with dump_path.open("r", encoding="utf8", errors="ignore") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            kmer, cnt = parts
            try:
                kmers[kmer] = int(cnt)
            except ValueError:
                continue
    return kmers


def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
    """Aggregate presence counts for each k-mer across a list of genomes.

    Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

    Parameters:
    - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
    - genome_ids: list of genome identifiers to include.

    Returns:
    - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
    """
    prevalence: Counter = Counter()
    dump_dir = Path(dump_dir)
    for gid in genome_ids:
        dump_path = dump_dir / f"{gid}_db_kmers.txt"
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        prevalence.update(kmers.keys())
    return prevalence


def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
                               min_frac: float = 0.02, max_frac: float = 0.95) -> list[str]:
    """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

    Parameters:
    - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
    - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
    - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
    - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

    Returns:
    - List of k-mer strings that pass the prevalence filters.
    """
    min_count = int(np.ceil(min_frac * n_genomes))
    max_count = int(np.floor(max_frac * n_genomes))
    vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
    return vocab


def build_sparse_matrix(
    dump_dir: str,
    genome_ids: list[str],
    vocab: list[str],
    binary: bool = True,
    chunk_size: int = 100,
    ) -> tuple[sparse.csr_matrix, list[str]]:
    """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

    This reads each genome's dump and populates the matrix using the provided `vocab` index.
    When `binary` is True, presence is recorded as 1; otherwise counts are used (int).
    The build is chunked to keep peak memory low.

    Parameters:
    - dump_dir: directory with per-genome k-mer dump files.
    - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
    - vocab: ordered list of k-mers corresponding to columns in the matrix.
    - binary: whether to collapse counts to binary presence/absence.
    - chunk_size: number of genomes per chunk when building the matrix.

    Returns:
    - (csr_matrix, kept_genome_ids) where kept_genome_ids excludes genomes with missing dumps.
    """
    vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
    dump_dir = Path(dump_dir)
    blocks: list[sparse.csr_matrix] = []
    kept_genome_ids: list[str] = []
    dtype = np.int8 if binary else np.int32
    n_features = len(vocab)

    for start in range(0, len(genome_ids), chunk_size):
        chunk_ids = genome_ids[start:start + chunk_size]
        present_ids = [gid for gid in chunk_ids if (dump_dir / f"{gid}_db_kmers.txt").exists()]
        if not present_ids:
            continue

        rows: list[int] = []
        cols: list[int] = []
        data: list[int] = []
        for row_idx, gid in enumerate(present_ids):
            dump_path = dump_dir / f"{gid}_db_kmers.txt"
            kmers = iter_genome_kmers(dump_path)
            for kmer in kmers.keys():
                col_idx = vocab_index.get(kmer)
                if col_idx is None:
                    continue
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(1 if binary else int(kmers.get(kmer, 0)))

        block = sparse.csr_matrix((data, (rows, cols)), shape=(len(present_ids), n_features), dtype=dtype)
        blocks.append(block)
        kept_genome_ids.extend(present_ids)

    if not blocks:
        return sparse.csr_matrix((0, n_features), dtype=dtype), []
    return sparse.vstack(blocks, format='csr'), kept_genome_ids


def load_labels(labels_path: Path) -> pd.Series:
    """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

    Expects a CSV with at least `Genome ID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

    Parameters:
    - labels_path: Path to CSV containing labels.

    Returns:
    - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
    """
    df = pd.read_csv(labels_path)

    # Normalize column lookup so small header variations do not break the notebook.
    norm = {c.strip().lower(): c for c in df.columns}
    gid_col = norm.get("genome id")
    pheno_col = norm.get("phenotype")

    if gid_col is None or pheno_col is None:
        raise ValueError(
            "Expected columns for Genome ID and phenotype in labels file. "
            f"Found columns: {list(df.columns)}"
        )

    pheno = df.set_index(gid_col)[pheno_col]
    pheno = pd.to_numeric(pheno, errors="coerce")
    return pheno

eval = {'model':[], 'bal_acc':[], 'acc':[], 'precision':[], 'recall':[], 'f1':[], 'f1_macro':[], 'roc_auc':[], 'auprc':[],} 
def metrics(model: str, metric: list):
    ''' Expects a list of names of all metrics neeeded, 
        Returns mean +/-  std for all metrics given '''
        
    bal_acc = f"{np.mean(metric['test_balanced_accuracy']):.4f} (+/- {np.std(metric['test_balanced_accuracy']):.4f})"
    acc = f"{np.mean(metric['test_accuracy']):.4f} (+/- {np.std(metric['test_accuracy']):.4f})"
    precision = f"{np.mean(metric['test_precision']):.4f} (+/- {np.std(metric['test_precision']):.4f})"
    recall = f"{np.mean(metric['test_recall']):.4f} (+/- {np.std(metric['test_recall']):.4f})"
    f1 = f"{np.mean(metric['test_f1']):.4f} (+/- {np.std(metric['test_f1']):.4f})"
    f1_macro = f"{np.mean(metric['test_f1_macro']):.4f} (+/- {np.std(metric['test_f1_macro']):.4f})"
    roc_auc = f"{np.mean(metric['test_roc_auc']):.4f} (+/- {np.std(metric['test_roc_auc']):.4f})"
    auprc = f"{np.mean(metric['test_average_precision']):.4f} (+/- {np.std(metric['test_average_precision']):.4f})"
    
    # save metrics. 
    eval['model'].extend(model)
    eval['model'].extend(bal_acc)
    eval['model'].extend(acc)
    eval['model'].extend(precision)
    eval['model'].extend(recall)
    eval['model'].extend(f1)
    eval['model'].extend(f1_macro)
    eval['model'].extend(roc_auc)
    eval['model'].extend(auprc)

    
    # print metrics
    print(f'bal_acc: {bal_acc}, acc: {acc}, precision:{precision}, recall: {recall} \n f1: {f1}, f1_macro: {f1_macro}, roc_auc: {roc_auc}, auprc: {auprc}')

## Run Feature selection and Train models
Adjust the paths below (`dump_dir`, `labels_path`, `genome_ids_path`) if your files are elsewhere, then run this cell.

In [3]:
# load labels
labels_path = Path('../data/phenotype/cefuroxime_phenotype.csv')
labels = load_labels(labels_path)
labels = labels.dropna()
labels = labels.astype(float)

In [4]:
# # get a Sample of genomes
# seed = 42
# from pathlib import Path
# import random

# labels = load_labels(labels_path)
# labels = labels.dropna()
# labels = labels.astype(float)

# res_ids = labels[labels == 1.0].index.astype(str).tolist()
# sus_ids = labels[labels == 0.0].index.astype(str).tolist()

# n_per_class = 1500
# if len(res_ids) < n_per_class or len(sus_ids) < n_per_class:
#     raise ValueError(f'Not enough genomes to sample: have {len(res_ids)} R, {len(sus_ids)} S')

# random.seed(seed)
# sampled_res = random.sample(res_ids, n_per_class)
# sampled_sus = random.sample(sus_ids, n_per_class)
# sampled_ids = sampled_res + sampled_sus

# # out_ids_path = Path('../data/phenotype/ampicillin_1000_ids.txt')
# # out_ids_path.parent.mkdir(parents=True, exist_ok=True)
# # with out_ids_path.open('w') as fh:
# #     fh.write('\n'.join(sampled_ids))

# # print(f'Sampled {len(sampled_ids)} genomes (R={n_per_class}, S={n_per_class}), saved to {out_ids_path}')




In [5]:
# read sample ids
with open ('../data/sampled_data/cefuroxime_1444_ids.txt', 'r') as data:
    sampled_ids = [d.strip() for d in data]

In [ ]:
# Build prevalence and vocabulary (prevalence filter)
dump_dir = Path('../data/counted_kmers')
prevalence = build_prevalence(dump_dir, sampled_ids)
print('Unique k-mers seen:', len(prevalence))

In [ ]:

# Filter by prevalence (1% - 99%) and cap vocab size if too large
vocab = select_vocab_by_prevalence(prevalence, n_genomes=len(sampled_ids), min_frac=0.01, max_frac=0.99)
print('Vocab after prevalence filter:', len(vocab))


### save vocab, so as not to build everytime

In [ ]:
# Save vocab (raw) for reproducibility
fv_dir = Path('../output/feature_selection')
fv_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(vocab, fv_dir / 'cefuroxime_vocab.joblib')
# with (fv_dir / 'cefuroxime_vocab_raw.txt').open('w', encoding='utf8') as fh:
#     fh.write('\n'.join(vocab))
print('Saved sampled ids and raw vocab.')

In [6]:
# Load saved vocab
fv_dir = Path('../output/feature_selection')
dump_dir = Path('../data/counted_kmers')
vocab = joblib.load(fv_dir / 'cefuroxime_vocab.joblib')


In [7]:
# Build binary presence sparse matrix (rows ordered as sampled_ids)
X, kept_ids = build_sparse_matrix(dump_dir, sampled_ids, vocab, binary=True, chunk_size=100)
print('Built X shape:', X.shape)
print('Kept genomes with dump files:', len(kept_ids))

# Build label vector aligned to the genomes that actually had dump files
# Ensure labels index is string-typed to match genome IDs
labels_str = labels.copy()
labels_str.index = labels_str.index.astype(str)

missing = [gid for gid in kept_ids if gid not in labels_str.index]
if missing:
    raise KeyError(f'Some kept ids are missing in labels: {missing[:5]}... total {len(missing)}')

sampled_ids = kept_ids
y = np.array([labels_str.loc[gid] for gid in sampled_ids], dtype=int)
print('Built y shape:', y.shape)


Built X shape: (1364, 521970)
Kept genomes with dump files: 1364
Built y shape: (1364,)


In [8]:
# Split data into train and held out validation set
from sklearn.model_selection import train_test_split 
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.20, random_state=42, stratify=y)
print(X_train.shape,  X_val.shape)

(1091, 521970) (273, 521970)


## Model Training

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression

# define global params
# number of K features to select
k_features = 20000 

# cv splits
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# metrics of interest
scoring = ['balanced_accuracy','accuracy', 'precision', 'recall', 'f1', 'f1_macro', 'roc_auc', 'average_precision']



##### Logistic Regression Model

In [10]:

pipeline = Pipeline([
    ('selectk', SelectKBest(chi2, k=min(k_features, X.shape[1]))),
    ('scaler', MaxAbsScaler()),  # MaxAbsScaler: This scales data but keeps sparse matrices intact 
    ('lr', LogisticRegression(
        penalty='elasticnet',     # aggressive regularization (ElasticNet) to force non-informative features to exactly 0
        l1_ratio=0.5,       # Balances between keeping groups of features (L2) and dropping them (L1)
        C=0.1,              # Strong penalty. Lower values = fewer features kept.
        solver='saga', 
        class_weight='balanced', 
        max_iter=5000,      # Increased to ensure the complex solver converges
        n_jobs=1,
        random_state=42
    ))
])


scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
print(scores)
metrics('LR', scores )

{'fit_time': array([220.71756792, 165.03619456, 149.92306185, 165.06216908,
       174.03016591]), 'score_time': array([0.96989584, 0.69540119, 0.66101742, 0.72439265, 0.76522231]), 'test_balanced_accuracy': array([0.68943944, 0.71069024, 0.75252525, 0.74686369, 0.74269597]), 'test_accuracy': array([0.68949772, 0.71100917, 0.75229358, 0.74770642, 0.74311927]), 'test_precision': array([0.68518519, 0.72277228, 0.73684211, 0.76530612, 0.74757282]), 'test_recall': array([0.68518519, 0.67592593, 0.77777778, 0.70093458, 0.71962617]), 'test_f1': array([0.68518519, 0.69856459, 0.75675676, 0.73170732, 0.73333333]), 'test_f1_macro': array([0.68943944, 0.71051578, 0.75221015, 0.74680604, 0.74277286]), 'test_roc_auc': array([0.76760093, 0.80959596, 0.83181818, 0.81350509, 0.77123853]), 'test_average_precision': array([0.73115167, 0.80767505, 0.84563806, 0.81499277, 0.76454243])}
bal_acc: 0.7284 (+/- 0.0243), acc: 0.7287 (+/- 0.0244), precision:0.7315 (+/- 0.0270), recall: 0.7119 (+/- 0.0361) 
 f1:

## Hyperparameter tuning on LR model

In [11]:
# Manually find the correct decision threshold for the model by evaluating the precision-recall curve on the training set.
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Fit the pipeline on a train/test split to analyze thresholds
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipeline.fit(X_train, y_train)

# Get raw probabilities instead of hard 0/1 predictions
y_prob = pipeline.predict_proba(X_val)[:, 1]

# Test thresholds from 0.3 to 0.7 to find where ALL metrics cross 80%
print(f"{'Threshold':<10}{'Accuracy':<10}{'Precision':<10}{'Recall':<10}{'F1-Score':<10}")
print("-" * 50)

for threshold in np.arange(0.3, 0.7, 0.02):
    y_pred_thresh = (y_prob >= threshold).astype(int)
     
    acc = accuracy_score(y_val, y_pred_thresh)
    prec = precision_score(y_val, y_pred_thresh)
    rec = recall_score(y_val, y_pred_thresh)
    f1 = f1_score(y_val, y_pred_thresh)
    
    print(f"{threshold:.2f}       {acc:.3f}     {prec:.3f}     {rec:.3f}     {f1:.3f}")

Threshold Accuracy  Precision Recall    F1-Score  
--------------------------------------------------
0.30       0.692     0.653     0.807     0.722
0.32       0.689     0.660     0.763     0.708
0.34       0.685     0.660     0.748     0.701
0.36       0.681     0.662     0.726     0.693
0.38       0.678     0.664     0.704     0.683
0.40       0.681     0.669     0.704     0.686
0.42       0.685     0.676     0.696     0.686
0.44       0.700     0.696     0.696     0.696
0.46       0.696     0.697     0.681     0.689
0.48       0.696     0.700     0.674     0.687
0.50       0.696     0.703     0.667     0.684
0.52       0.681     0.697     0.630     0.661
0.54       0.692     0.722     0.615     0.664
0.56       0.696     0.741     0.593     0.658
0.58       0.700     0.752     0.585     0.658
0.60       0.696     0.755     0.570     0.650
0.62       0.685     0.753     0.541     0.629
0.64       0.689     0.766     0.533     0.629
0.66       0.670     0.759     0.489     0.595
0.68 

In [16]:
# best threshold and metrics
best_threshold_lr = 0.44
y_pred_thresh = (y_prob >= best_threshold_lr).astype(int)
acc = accuracy_score(y_val, y_pred_thresh)
prec = precision_score(y_val, y_pred_thresh)
rec = recall_score(y_val, y_pred_thresh)
f1 = f1_score(y_val, y_pred_thresh)
print(f'best threshold: {best_threshold_lr}')
print(f'acc: {acc:.3f}, precision: {prec:.3f}, recall: {rec:.3f}, f1: {f1:.3f}')

best threshold: 0.44
acc: 0.700, precision: 0.696, recall: 0.696, f1: 0.696


In [ ]:
# # Using GridSearchCV to tune
# from sklearn.model_selection import GridSearchCV

# # Define a hyperparameter grid around your current setup
# param_grid = {
#         # Test if fewer features reduces noise better
#     'lr__C': [0.01, 0.1, 1.0],             # Regularization strength
# # Balance between L1 and L2 penalties
# }

# # Run the search optimizing directly for your goal metric (F1-score)
# grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=-1, verbose=1)
# grid_search.fit(X, y)

# print(f"Best parameters found: {grid_search.best_params_}")
# print(f"Best Cross-Validated Balanced Accuracy: {grid_search.best_score_:.4f}")

##### LightGBM Model

In [ ]:
# building lightgbm model
import lightgbm as lgb

# Tree-based models don't need scaling, and they handle high dimensions very fast
lgb_pipeline = Pipeline([
    ('selectk', SelectKBest(chi2, k=k_features)),
    ('lgb', lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.01,       # Slower learning rate prevents rapid overfitting
        num_leaves=15,            # Smaller trees (forces it to find broader patterns)
        max_depth=5,              # Shallower trees
        min_child_samples=40,     # A leaf must have at least 40 bacteria (stops it from memorizing rare clones)
        reg_alpha=0.5,            # L1 Regularization (drops noisy k-mers)
        reg_lambda=0.5,           # L2 Regularization (smooths weights)
        class_weight='balanced',
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    )
])

# Convert sparse matrix to float32 for LightGBM compatibility
X_lgb = X_train.astype(np.float32)
# Evaluate LightGBM
scores_lgb = cross_validate(lgb_pipeline, X_lgb, y_train, cv=cv, scoring=scoring)
print(scores_lgb)
metrics('LightGBM', scores_lgb)

c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ut

{'fit_time': array([37.692312  , 33.89723468, 31.35762668, 27.13325477, 29.855968  ]), 'score_time': array([2.36486101, 1.87346983, 1.52853513, 1.38457918, 1.62973666]), 'test_balanced_accuracy': array([0.69406907, 0.76161616, 0.74772727, 0.73785468, 0.74303275]), 'test_accuracy': array([0.69406393, 0.76146789, 0.74770642, 0.73853211, 0.74311927]), 'test_precision': array([0.68807339, 0.75      , 0.74311927, 0.75      , 0.73831776]), 'test_recall': array([0.69444444, 0.77777778, 0.75      , 0.70093458, 0.73831776]), 'test_f1': array([0.69124424, 0.76363636, 0.74654378, 0.72463768, 0.73831776]), 'test_f1_macro': array([0.69403841, 0.76144781, 0.74770111, 0.73786469, 0.74303275]), 'test_roc_auc': array([0.76901902, 0.84351852, 0.83013468, 0.8040751 , 0.79220342]), 'test_average_precision': array([0.7746999 , 0.83863917, 0.85003714, 0.82305689, 0.79867543])}
bal_acc: 0.7369 (+/- 0.0228), acc: 0.7370 (+/- 0.0228), precision:0.7339 (+/- 0.0233), recall: 0.7323 (+/- 0.0311) 
 f1: 0.7329 (+/-

In [35]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import lightgbm as lgb

# Get the raw cross-validated probabilities from your LightGBM pipeline
y_prob_lgb = cross_val_predict(lgb_pipeline, X_lgb, y_train, cv=cv, method='predict_proba')[:, 1]

print(f"{'Threshold':<10}{'Accuracy':<10}{'Precision':<10}{'Recall':<10}{'F1-Score':<10}")
print("-" * 50)

# Scan thresholds
for threshold in np.arange(0.3, 0.7, 0.02):
    y_pred_thresh = (y_prob_lgb >= threshold).astype(int)
    
    acc = accuracy_score(y_train, y_pred_thresh)
    prec = precision_score(y_train, y_pred_thresh)
    rec = recall_score(y_train, y_pred_thresh)
    f1 = f1_score(y_train, y_pred_thresh)
    
    print(f"{threshold:.2f}       {acc:.3f}     {prec:.3f}     {rec:.3f}     {f1:.3f}")

c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ut

Threshold Accuracy  Precision Recall    F1-Score  
--------------------------------------------------
0.30       0.682     0.630     0.862     0.728
0.32       0.686     0.636     0.848     0.727
0.34       0.696     0.648     0.838     0.731
0.36       0.709     0.664     0.831     0.738
0.38       0.709     0.669     0.812     0.734
0.40       0.722     0.686     0.807     0.741
0.42       0.726     0.694     0.796     0.741
0.44       0.731     0.704     0.783     0.741
0.46       0.735     0.717     0.766     0.740
0.48       0.738     0.727     0.749     0.738
0.50       0.737     0.734     0.732     0.733
0.52       0.743     0.751     0.717     0.734
0.54       0.742     0.762     0.695     0.727
0.56       0.741     0.771     0.675     0.720
0.58       0.741     0.779     0.662     0.716
0.60       0.738     0.796     0.630     0.703
0.62       0.736     0.805     0.613     0.696
0.64       0.733     0.817     0.591     0.686
0.66       0.728     0.825     0.569     0.673
0.68 

In [32]:
# best threshold and metrics
best_threshold_lgb = 0.48
y_pred_thresh = (y_prob_lgb >= best_threshold_lgb).astype(int)
acc = accuracy_score(y_train, y_pred_thresh)
prec = precision_score(y_train, y_pred_thresh)
rec = recall_score(y_train, y_pred_thresh)
f1 = f1_score(y_train, y_pred_thresh)
print(f'best threshold: {best_threshold_lgb}')
print(f'acc: {acc:.3f}, precision: {prec:.3f}, recall: {rec:.3f}, f1: {f1:.3f}')

best threshold: 0.48
acc: 0.739, precision: 0.725, recall: 0.757, f1: 0.741


#### Performing Error Analysis 

In [ ]:
# import pandas as pd
# from sklearn.model_selection import cross_val_predict

# # 1. Get the exact probabilities from the LightGBM pipeline
# # (Ensure your X matrix is already float32)
# # y_prob_cv = cross_val_predict(lgb_pipeline, X, y, cv=cv, method='predict_proba')[:, 1]

# # 2. Build the error analysis dataframe
# error_df = pd.DataFrame({
#     'GenomeID': sampled_ids,
#     'True_Label': y,
#     'Predicted_Prob': y_prob_lgb
# })

# # 3. Calculate Error Magnitude (How far off was the probability from the truth?)
# error_df['Error_Magnitude'] = abs(error_df['True_Label'] - error_df['Predicted_Prob'])

# # 4. Sort to find the genomes the model was most confidently wrong about
# worst_mistakes = error_df.sort_values(by='Error_Magnitude', ascending=False)

# print("--- The Top 15 Most Confident Mistakes ---")
# print(worst_mistakes.head(15).to_string(index=False))

## Build Final Model and evaluating on held out set

In [36]:
# Using best performing algorithm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score
model = lgb_pipeline.fit(X_lgb, y_train)

predicted = model.predict(X_val.astype(np.float32))
bal_acc = balanced_accuracy_score(y_val, predicted)
acc = accuracy_score(y_val, predicted)
prec = precision_score(y_val, predicted)
rec = recall_score(y_val, predicted)
f1 = f1_score(y_val, predicted)
f1_macro = f1_score(y_val, predicted, average='macro')
roc_auc = roc_auc_score(y_val, predicted)

print(f'bal_acc: {bal_acc:.3f}, acc: {acc:.3f}, precision: {prec:.3f},\n rec: {rec:.3f}, f1: {f1:.3f}, f1_macro: {f1_macro:.3f}, roc_auc: {roc_auc:.3f} ')

bal_acc: 0.692, acc: 0.692, precision: 0.698,
 rec: 0.667, f1: 0.682, f1_macro: 0.692, roc_auc: 0.692 


c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [41]:
predict = model.predict_proba(X_val.astype(np.float32))[:,1]
predicted = (predict >= 0.5).astype(int)
bal_acc = balanced_accuracy_score(y_val, predicted)
acc = accuracy_score(y_val, predicted)
prec = precision_score(y_val, predicted)
rec = recall_score(y_val, predicted)
f1 = f1_score(y_val, predicted)
f1_macro = f1_score(y_val, predicted, average='macro')
roc_auc = roc_auc_score(y_val, predicted)
print(f'bal_acc: {bal_acc:.3f}, acc: {acc:.3f}, precision: {prec:.3f},\n rec: {rec:.3f}, f1: {f1:.3f}, f1_macro: {f1_macro:.3f}, roc_auc: {roc_auc:.3f} ')

bal_acc: 0.692, acc: 0.692, precision: 0.698,
 rec: 0.667, f1: 0.682, f1_macro: 0.692, roc_auc: 0.692 


c:\Users\Mr_Nnobody\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:

# # Save pipeline
# out_models = Path('../output/models')
# joblib.dump(model, out_models+'/lr_pipeline')
# print('Saved model to', out_models)


In [ ]:
# Extract learned features
all_coefs = []

#extract coeffs
coefs = lgb_pipeline.named_steps['lgb'].coef_[0]
all_coefs.append(coefs)

# average importance across all folds
mean_importance = np.mean(np.abs(all_coefs), axis=0)
# get indices of the top 100 most stable kmers 
top_kmer_indices =np.argsort(mean_importance)[-100:]

In [ ]:
# TODO: plot feature of importance for top features.